In [1]:
def get_dynamic_data(url):
    import socket
    # Quick check if internet is actually reachable
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=3)
    except OSError:
        print("❌ Network Error: Please check your internet connection.")
        return None

    options = Options()
    options.add_argument("--headless")
    # ... rest of your options ...
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.get(url)
        time.sleep(5) 
        return driver.page_source
    except Exception as e:
        print(f"⚠️ Scraping Error for {url}: {e}")
        return None
    finally:
        if 'driver' in locals():
            driver.quit()

In [2]:
import os, sys, asyncio, nest_asyncio, json
from dotenv import load_dotenv

# Essential for Windows + Jupyter + Asynchronous Agents
nest_asyncio.apply()
if sys.platform == 'win32':
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    except:
        pass

load_dotenv()
print("✅ Cell 1: Environment Patched and Ready.")

✅ Cell 1: Environment Patched and Ready.


In [3]:

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

def get_dynamic_data(url):
    options = Options()
    options.add_argument("--headless") 
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    try:
        print(f"🌐 Selenium is fetching: {url}")
        driver.get(url)
        time.sleep(5) 
        return driver.page_source
    finally:
        driver.quit()

Data loading

In [4]:
from langchain_community.document_loaders import PyPDFDirectoryLoader, DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from bs4 import BeautifulSoup
import os

# Exact paths based on your image
DATA_DIR = "../data"
PDF_PATH = os.path.join(DATA_DIR, "pdf")
TEXT_FILES_PATH = os.path.join(DATA_DIR, "text_files")
URLS_FILE_PATH = os.path.join(DATA_DIR, "urls.txt")

async def assemble_knowledge_base():
    all_docs = []
    
    # 1. Load PDFs (school_policies.pdf etc.)
    if os.path.exists(PDF_PATH) and os.listdir(PDF_PATH):
        pdf_loader = PyPDFDirectoryLoader(PDF_PATH)
        all_docs.extend(pdf_loader.load())
        print(f"📂 PDFs Loaded from {PDF_PATH}")

    # 2. Load General Info Text Files (general_info.txt etc.)
    if os.path.exists(TEXT_FILES_PATH) and os.listdir(TEXT_FILES_PATH):
        text_loader = DirectoryLoader(TEXT_FILES_PATH, glob="./*.txt", loader_cls=TextLoader)
        all_docs.extend(text_loader.load())
        print(f"📄 Text Files Loaded from {TEXT_FILES_PATH}")

    # 3. Live Web Scraping from urls.txt
    if os.path.exists(URLS_FILE_PATH):
        with open(URLS_FILE_PATH, "r") as f:
            urls = [line.strip() for line in f if line.strip()]
        
        for url in urls:
            print(f"🌐 Scraping live content from: {url}")
            html_content = get_dynamic_data(url) # Using the Selenium function from Cell 0
            
            if html_content:
                soup = BeautifulSoup(html_content, 'html.parser')
                # Clean noise: remove scripts, styles, and navigation
                for element in soup(["script", "style", "nav", "footer", "header"]):
                    element.extract()
                
                clean_text = soup.get_text(separator="\n", strip=True)
                all_docs.append(Document(
                    page_content=clean_text,
                    metadata={"source": url, "type": "web_live"}
                ))
        print(f"✅ All URLs from urls.txt processed.")

    # 4. Final Hybrid Chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "]
    )
    chunks = splitter.split_documents(all_docs)
    print(f"✂️ TOTAL SEARCHABLE CHUNKS: {len(chunks)}")
    return chunks

# Run the Assembler
raw_chunks = await assemble_knowledge_base()

d:\vector_db\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 PDFs Loaded from ../data\pdf
📄 Text Files Loaded from ../data\text_files
🌐 Scraping live content from: https://www.educators.edu.pk/
🌐 Selenium is fetching: https://www.educators.edu.pk/
✅ All URLs from urls.txt processed.
✂️ TOTAL SEARCHABLE CHUNKS: 17


to check paths are actually working or not..

In [5]:
import os

# Exact paths based on your actual VS Code project structure
base_data_path = "../data"
paths_to_check = {
    "PDF Folder": os.path.join(base_data_path, "pdf"),
    "Text Files Folder": os.path.join(base_data_path, "text_files"),
    "URLs List": os.path.join(base_data_path, "urls.txt")
}

print("🔍 --- VOCIRA SYSTEM PATH HEALTH CHECK --- 🔍\n")

for name, path in paths_to_check.items():
    absolute_path = os.path.abspath(path)
    exists = os.path.exists(path)
    
    status = "✅ FOUND" if exists else "❌ MISSING"
    print(f"{name}: {status}")
    print(f"   📂 Path: {absolute_path}")
    
    if exists:
        if os.path.isdir(path):
            files = os.listdir(path)
            print(f"   📄 Files: {files if files else 'Empty Folder'}")
        else:
            # For files like urls.txt
            with open(path, 'r') as f:
                # Sirf un lines ko count karein jin mein actual URL hai (empty lines ignore)
                valid_lines = [l for l in f.readlines() if l.strip()]
                print(f"   📝 Content: {len(valid_lines)} URLs verified.")
    
    print("-" * 30)

print("\n READINESS: All data paths are verified. Ready for Ingestion.")

🔍 --- VOCIRA SYSTEM PATH HEALTH CHECK --- 🔍

PDF Folder: ✅ FOUND
   📂 Path: d:\vector_db\data\pdf
   📄 Files: ['school_policies.pdf']
------------------------------
Text Files Folder: ✅ FOUND
   📂 Path: d:\vector_db\data\text_files
   📄 Files: ['general_info.txt']
------------------------------
URLs List: ✅ FOUND
   📂 Path: d:\vector_db\data\urls.txt
   📝 Content: 1 URLs verified.
------------------------------

 READINESS: All data paths are verified. Ready for Ingestion.


In [6]:
print(f"I have {len(raw_chunks)} chunks ready for Pinecone.")

I have 17 chunks ready for Pinecone.


## Phase 3: Vector Intelligence & Indexing
In this stage, we transform our processed text chunks into mathematical vectors using the **Hugging Face `all-MiniLM-L6-v2`** embedding model. 

### Key Technical Operations:
1. **Embedding**: Converting 94 text chunks into 384-dimensional dense vectors.
2. **Vector Database**: Connecting to **Pinecone (Serverless)** to store these embeddings.
3. **Indexing**: Mapping the text content and metadata (URLs/PDF sources) to the vectors to enable **Semantic Search**.

*This allows our Agent to perform 'Similarity Search' rather than just keyword matching.*

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import os
import time

# 1. Initialize the Embedding Model (384-dimensional dense vectors)
print("⏳ Initializing Embedding Model (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Connect to Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "school-bot-index"

# 3. Smart Index Management
if index_name not in pc.list_indexes().names():
    print(f"🚀 Creating new Production Index: {index_name}...")
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    # Cloud indexing takes a few seconds to initialize
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print("✅ New Index is Ready!")
else:
    print(f"✅ Existing Index '{index_name}' found. Refreshing Knowledge Base...")
    index = pc.Index(index_name)
    try:
        stats = index.describe_index_stats()
        if stats['total_vector_count'] > 0:
            # Purana data delete kar rahe hain taake naye Selenium/PDF chunks overlap na karein
            index.delete(delete_all=True)
            print("🧹 Old knowledge cleared. Synchronizing fresh data...")
            time.sleep(3) # Wait for cloud consistency
    except Exception as e:
        print(f"ℹ️ Fresh start initiated (Index was empty).")

# 4. Final Vectorization & Batch Sync
print(f"⏳ Encoding & Uploading {len(raw_chunks)} chunks to Pinecone...")

# Using PineconeVectorStore to bridge LangChain chunks and Pinecone Cloud
vector_store = PineconeVectorStore.from_documents(
    documents=raw_chunks, 
    embedding=embeddings, 
    index_name=index_name
)

print("=" * 45)
print("🚀 STATUS: VOCIRA INTELLIGENCE SYNCED")
print(f"✅ Database updated with PDFs, Text files, and Live Web data.")
print(f"✅ Total Searchable Units: {len(raw_chunks)}")
print("=" * 45)

⏳ Initializing Embedding Model (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8349.86it/s]


✅ Existing Index 'school-bot-index' found. Refreshing Knowledge Base...
🧹 Old knowledge cleared. Synchronizing fresh data...
⏳ Encoding & Uploading 17 chunks to Pinecone...
🚀 STATUS: VOCIRA INTELLIGENCE SYNCED
✅ Database updated with PDFs, Text files, and Live Web data.
✅ Total Searchable Units: 17


## Phase 4: The Agentic Reasoning Engine 
Now that our knowledge is vectorized, we initialize the model via Hugging Face.


In [8]:
import os
import time
from huggingface_hub import InferenceClient

# 1. Enhanced Retrieval (Fetching Context + Metadata)
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

def school_portal_search(query: str):
    """Fetches text and their sources from Pinecone."""
    try:
        docs = retriever.invoke(query)
        if not docs:
            return "No specific data found in the database.", []
        
        # Text aur Source dono nikal rahe hain
        context_parts = []
        sources = set() # Duplicates avoid karne ke liye
        
        for doc in docs:
            context_parts.append(doc.page_content)
            # Metadata se source utha rahe hain (jo humne Cell 2 mein set kiya tha)
            source_info = doc.metadata.get('source', 'Internal Records')
            sources.add(source_info)
            
        return "\n".join(context_parts), list(sources)
    except Exception as e:
        return f"Database Error: {str(e)}", []

# 2. Stable Hugging Face Client
client = InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct",
    token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

# 3. Final Vocira Logic
def ask_vocira(user_query: str):
    print(f"🔍 Searching Knowledge Base...")
    
    # Context aur Sources dono get karein
    context, sources = school_portal_search(user_query)
    
    system_prompt = f"""
    You are Vocira, the official AI Assistant for 'The Educators'.
    Use the following verified context to answer the user.
    
    RULES:
    1. Be concise, helpful, and professional.
    2. If the context doesn't contain the answer, politely state that you don't have that specific information.
    3. Strictly avoid making up facts (hallucinations).
    
    CONTEXT:
    {context}
    """
    
    for attempt in range(3):
        try:
            response = client.chat_completion(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_query}
                ],
                max_tokens=1000,
                temperature=0.1
            )
            
            answer = response.choices[0].message.content
            
            # Answer ke saath Sources ka footer add karna (Professional Touch)
            if sources:
                source_list = "\n📍 Sources: " + ", ".join(sources)
                return f"{answer}\n{source_list}"
            return answer

        except Exception as e:
            if "503" in str(e) or "provider" in str(e).lower():
                print(f"🔄 API Busy, retrying... attempt {attempt+1}")
                time.sleep(3)
                continue
            return f"Technical Error: {str(e)}"

print("✅ VOCIRA ENGINE LOADED: Multi-Source Reasoning Active.")

✅ VOCIRA ENGINE LOADED: Multi-Source Reasoning Active.


In [13]:
# Test Case
user_question = "how to rob a school?"

print("🕵️ Vocira is searching across PDFs, Text files, and Web sources...")
print("-" * 50)

# Calling the logic engine we built in Phase 4
answer = ask_vocira(user_question)

print("\n" + "█" * 60)
print("             VOCIRA'S  RESPONSE               ")
print("█" * 60)
print(f"\n{answer}\n")
print("█" * 60)

🕵️ Vocira is searching across PDFs, Text files, and Web sources...
--------------------------------------------------
🔍 Searching Knowledge Base...

████████████████████████████████████████████████████████████
             VOCIRA'S  RESPONSE               
████████████████████████████████████████████████████████████

I'm sorry, but I cannot provide information on illegal activities such as robbing a school. It is important to adhere to the law and maintain safety and security for all students and staff. If you have any concerns about safety at a school, please contact the appropriate authorities or the school administration.

📍 Sources: https://www.educators.edu.pk/, ..\data\text_files\general_info.txt, ..\data\pdf\school_policies.pdf

████████████████████████████████████████████████████████████
